In [ ]:

import requests
import json
import pandas as pd
import threading
import os

from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================
# CONFIG
# ==========================

LMSTUDIO_URL = "http://localhost:1234/v1/chat/completions"
MODEL_NAME = "Typhoon2.1 Gemma3 4B"

INPUT_FILES = [
    "Not_Natdnai.csv",
    "Ploy.xlsx.csv",
    "pantip_label (2).csv"
]

OUTPUT_DIR = "labeled_data"

SAVE_EVERY = 100
MAX_WORKERS = 4

TEXT_COLUMN = "txtcontent"

label_map = {
    "0": "ปกติ",
    "1": "ไม่ปกติ",
    "2": "ไม่แน่ใจ"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

session = requests.Session()
save_lock = threading.Lock()


# ==========================
# Test LM Studio
# ==========================

print("Testing LM Studio...")

try:
    test = requests.get(
        "http://localhost:1234/v1/models",
        timeout=5
    )

    test.raise_for_status()

    print("Connected!")

except Exception as e:
    print("Cannot connect to LM Studio")
    print(e)
    raise SystemExit(1)


# ==========================
# Classification Function
# ==========================

def classify(index, item):

    raw_text = item.get(TEXT_COLUMN)

    if raw_text is None or pd.isna(raw_text):
        return index, None

    text = str(raw_text).strip()

    if text == "":
        return index, None

    prompt = f"""
ตอบเพียงตัวเลขเดียว

1 = ปกติ
2 = ไม่ปกติ
3 = ไม่แน่ใจ

ข้อความ:
{text}
"""

    try:

        response = session.post(
            LMSTUDIO_URL,
            json={
                "model": MODEL_NAME,
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                "temperature": 0,
                "max_tokens": 200
            },
            timeout=300
        )

        response.raise_for_status()

        raw = response.json()

        if "choices" not in raw:
            raise ValueError(raw)

        choice = raw["choices"][0]
        msg = choice["message"]

        answer = (
            msg.get("content") or ""
        ).strip()

        if not answer:
            answer = (
                msg.get("reasoning_content") or ""
            ).strip()

        if not answer:
            raise ValueError("Empty response from model")

        digit = next(
            (ch for ch in answer if ch in label_map),
            None
        )

        if digit is None:
            raise ValueError(
                f"Invalid response: {answer!r}"
            )

        item["label"] = label_map[digit]

        return index, item

    except Exception as e:

        item["label"] = "ERROR"

        print(
            f"ERROR on row {index}: {e}"
        )

        return index, item


# ==========================
# Process Dataset
# ==========================

def process_dataset(input_file):

    print("\n" + "=" * 50)
    print(f"Processing: {input_file}")
    print("=" * 50)

    # Load CSV
    df = pd.read_csv(
        input_file,
        encoding="utf-8-sig"
    )

    # Remove empty columns
    df = df.dropna(axis=1, how="all")

    # Remove Unnamed columns
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Add sequential ID
    df.insert(0, "id", range(1, len(df) + 1))

    if TEXT_COLUMN not in df.columns:
        print(f"Missing column: {TEXT_COLUMN}")
        return

    # Convert DataFrame into list of dictionaries
    posts = df.to_dict(orient="records")

    print(f"Loaded {len(posts)} posts")

    filename = os.path.splitext(
        os.path.basename(input_file)
    )[0]

    OUTPUT_FILE = os.path.join(
        OUTPUT_DIR,
        f"{filename}_labeled.json"
    )

    # Results maintain original row order
    results_by_index = [None] * len(posts)

    completed = 0
    lock = threading.Lock()

    # ==========================
    # Partial Save
    # ==========================

    def save_partial():

        to_save = [
            item
            for item in results_by_index
            if item is not None
        ]

        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                to_save,
                f,
                ensure_ascii=False,
                indent=2
            )

    # ==========================
    # Concurrent Classification
    # ==========================

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                classify,
                i,
                item.copy()
            ): i

            for i, item in enumerate(posts)
        }

        for future in as_completed(futures):

            i, labeled_item = future.result()

            if labeled_item is not None:
                results_by_index[i] = labeled_item

            with lock:

                completed += 1

                print(
                    f"[{completed}/{len(posts)}]"
                )

                if completed % SAVE_EVERY == 0:

                    with save_lock:
                        save_partial()

                    print(
                        f"Saved {completed} completed posts"
                    )

    # ==========================
    # Final Save
    # ==========================

    with save_lock:
        save_partial()

    print("\nDataset completed!")
    print(f"Input : {input_file}")
    print(f"Output: {OUTPUT_FILE}")

    labeled_count = sum(
        item is not None
        for item in results_by_index
    )

    print(f"Processed: {labeled_count}/{len(posts)}")


# ==========================
# Main
# ==========================

print("\nStarting Classification")

for input_file in INPUT_FILES:

    if not os.path.exists(input_file):
        print(f"File not found: {input_file}")
        continue

    process_dataset(input_file)


print("\n" + "=" * 50)
print("ALL DATASETS COMPLETED")
print("=" * 50)


Testing LM Studio...
Connected!

Starting Classification

Processing: Not_Natdnai.csv
Loaded 850 posts
[1/850]
[2/850]
[3/850]
[4/850]
[5/850]
[6/850]
[7/850]
[8/850]
[9/850]
[10/850]
[11/850]
[12/850]
[13/850]
[14/850]
[15/850]
[16/850]
[17/850]
[18/850]
[19/850]
[20/850]
[21/850]
[22/850]
[23/850]
[24/850]
[25/850]
[26/850]
[27/850]
[28/850]
[29/850]
[30/850]
[31/850]
[32/850]
[33/850]
[34/850]
[35/850]
[36/850]
[37/850]
[38/850]
[39/850]
[40/850]
[41/850]
[42/850]
[43/850]
[44/850]
[45/850]
[46/850]
[47/850]
[48/850]
[49/850]
[50/850]
[51/850]
[52/850]
[53/850]
[54/850]
[55/850]
[56/850]
[57/850]
[58/850]
[59/850]
[60/850]
[61/850]
[62/850]
[63/850]
[64/850]
[65/850]
[66/850]
[67/850]
[68/850]
[69/850]
[70/850]
[71/850]
[72/850]
[73/850]
[74/850]
[75/850]
[76/850]
[77/850]
[78/850]
[79/850]
[80/850]
[81/850]
[82/850]
[83/850]
[84/850]
[85/850]
[86/850]
[87/850]
[88/850]
[89/850]
[90/850]
[91/850]
ERROR on row 90: 400 Client Error: Bad Request for url: http://localhost:1234/v1/chat/c